In [11]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [13]:
csv_path = "/content/drive/MyDrive/ML/abalone.csv"
df =pd.read_csv(csv_path)

print(df.head())
print(df.columns)

  Sex  Length  Diameter  Height  Whole weight  Shucked weight  Viscera weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  
Index(['Sex', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight',
       'Viscera weight', 'Shell weight', 'Rings'],
      dtype='object')


In [14]:
df['Sex'] = df['Sex'].map({'M': 0, 'F': 1, 'I': 2}) #label encoding of nominal feature

x = df.drop('Rings', axis=1)
y = df['Rings']
print(x.head())
print(x.shape)

   Sex  Length  Diameter  Height  Whole weight  Shucked weight  \
0    0   0.455     0.365   0.095        0.5140          0.2245   
1    0   0.350     0.265   0.090        0.2255          0.0995   
2    1   0.530     0.420   0.135        0.6770          0.2565   
3    0   0.440     0.365   0.125        0.5160          0.2155   
4    2   0.330     0.255   0.080        0.2050          0.0895   

   Viscera weight  Shell weight  
0          0.1010         0.150  
1          0.0485         0.070  
2          0.1415         0.210  
3          0.1140         0.155  
4          0.0395         0.055  
(4177, 8)


In [15]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature       #The feature index used for division
        self.threshold = threshold   #Division threshold value
        self.left = left             #left subtree
        self.right = right           #rightt subtree
        self.value = value           #If it is a leaf node, the predicted value

    def is_leaf(self):
        return self.value is not None

In [16]:
class DecisionTreeRegressorCustom:
    def __init__(self, max_depth=10, min_samples_split=5):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, x, y, rng):
        # Feature Subsampling
        self.n_features_to_split = int(np.ceil(x.shape[1] / 3))
        self.root = self._build_tree(x, y, depth=0, rng=rng)

    def _build_tree(self, x, y, depth, rng):
        n_samples, n_feats = x.shape

        #Stopping criteria: Leaf formation occurs if maximum depth is reached or sample size decreases.
        if (depth >= self.max_depth or
            n_samples < self.min_samples_split or
            np.all(y == y[0])):
            return Node(value=np.mean(y))

        # Selecting a random subset of features
        feat_idxs = rng.choice(n_feats, self.n_features_to_split, replace=False)

        # Finding the best split
        best_feat, best_thresh = self._best_split(x, y, feat_idxs)

        if best_feat is None:
            return Node(value=np.mean(y))

        # splitting data
        left_idxs = x[:, best_feat] <= best_thresh
        right_idxs = x[:, best_feat] > best_thresh

        #Creating subtrees recursively
        left_child = self._build_tree(x[left_idxs], y[left_idxs], depth + 1, rng)
        right_child = self._build_tree(x[right_idxs], y[right_idxs], depth + 1, rng)

        return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

    def _best_split(self, x, y, feat_idxs):
        best_variance_reduction = -1
        split_idx, split_thresh = None, None
        current_variance = np.var(y) * len(y)

        for feat in feat_idxs:
            X_column = x[:, feat]
            # Find unique threshold values
            thresholds = np.percentile(X_column, np.arange(10, 100, 10))

            for thresh in thresholds:
                left_y = y[X_column <= thresh]
                right_y = y[X_column > thresh]

                if len(left_y) == 0 or len(right_y) == 0:
                    continue

                # MSE loss after dividing
                left_var = np.var(left_y) * len(left_y)
                right_var = np.var(right_y) * len(right_y)
                variance_reduction = current_variance - (left_var + right_var)

                if variance_reduction > best_variance_reduction:
                    best_variance_reduction = variance_reduction
                    split_idx = feat
                    split_thresh = thresh

        return split_idx, split_thresh

    def predict(self, x):
        return np.array([self._traverse_tree(inputs, self.root) for inputs in x])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [17]:
class RandomForestRegressorCustom:
    def __init__(self, n_estimators=10, max_depth=8, min_samples_split=5, seed=1):
        self.n_estimators = n_estimators      # tree numbers
        self.max_depth = max_depth            # max depht of trees
        self.min_samples_split = min_samples_split
        self.seed = seed
        self.trees = []

    def fit(self, x, y):
        self.trees = []
        rng = np.random.default_rng(self.seed)
        n_samples = x.shape[0]

        for i in range(self.n_estimators):
            #Bootstrapping
            bootstrap_idxs = rng.choice(n_samples, n_samples, replace=True)
            x_b = x[bootstrap_idxs]
            y_b = y[bootstrap_idxs]

            #Training a new decision tree
            tree = DecisionTreeRegressorCustom(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            tree.fit(x_b, y_b, rng)
            self.trees.append(tree)
            print(f"Tree {i+1}/{self.n_estimators} trained...")
    def predict(self, x):
       # Tüm ağaçlardan gelen tahminleri toplayıp ortalamasını alıyoruz
        tree_preds = np.array([tree.predict(x) for tree in self.trees])
        return np.mean(tree_preds, axis=0)

In [18]:
def train_test_split_numpy(x, y, test_size =0.2, seed=1):
  rng = np.random.default_rng(seed)
  n = x.shape[0]
  idx =np.arange(n)
  rng.shuffle(idx)

  test_n =int(round(n*test_size))
  test_idx = idx[:test_n]
  train_idx= idx[test_n:]

  return x[train_idx], x[test_idx], y[train_idx], y[test_idx]

In [19]:
x_train, x_test, y_train, y_test = train_test_split_numpy(x.values, y.values, test_size=0.2, seed=1)

In [20]:
rf_model = RandomForestRegressorCustom(n_estimators=40, max_depth=10, min_samples_split=5, seed=1)
print("Random Forest model is training (so pls wait)...")
rf_model.fit(x_train, y_train)

y_rf_pred = rf_model.predict(x_test)

# R2 Score
def r2_score(y_true, y_pred):
    return 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))

# MAE
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

rf_r2 = r2_score(y_test, y_rf_pred)
rf_mae = mae(y_test, y_rf_pred)

print(f"Random Forest R2 Score: %{rf_r2 * 100:.2f}")
print(f"Random Forest MAE fault: {rf_mae:.4f}")

Random Forest model is training (so pls wait)...
Tree 1/40 trained...
Tree 2/40 trained...
Tree 3/40 trained...
Tree 4/40 trained...
Tree 5/40 trained...
Tree 6/40 trained...
Tree 7/40 trained...
Tree 8/40 trained...
Tree 9/40 trained...
Tree 10/40 trained...
Tree 11/40 trained...
Tree 12/40 trained...
Tree 13/40 trained...
Tree 14/40 trained...
Tree 15/40 trained...
Tree 16/40 trained...
Tree 17/40 trained...
Tree 18/40 trained...
Tree 19/40 trained...
Tree 20/40 trained...
Tree 21/40 trained...
Tree 22/40 trained...
Tree 23/40 trained...
Tree 24/40 trained...
Tree 25/40 trained...
Tree 26/40 trained...
Tree 27/40 trained...
Tree 28/40 trained...
Tree 29/40 trained...
Tree 30/40 trained...
Tree 31/40 trained...
Tree 32/40 trained...
Tree 33/40 trained...
Tree 34/40 trained...
Tree 35/40 trained...
Tree 36/40 trained...
Tree 37/40 trained...
Tree 38/40 trained...
Tree 39/40 trained...
Tree 40/40 trained...
Random Forest R2 Score: %56.01
Random Forest MAE fault: 1.4982
